# Data Engineering Analytics

This notebook reads the SQLite data generated by `pipeline_etl.py` and answers the required analytical questions.

In [17]:
import sqlite3
import pandas as pd

DB_PATH = "data/sunrise_sunset.db"
conn = sqlite3.connect(DB_PATH)

check_df = pd.read_sql_query("SELECT COUNT(*) AS total_rows FROM sunrise_sunset", conn)
check_df

,total_rows
0,2254


## 1) Longest and Shortest Day (Per Calendar Year)

In [18]:
base_df = pd.read_sql_query(
    "SELECT date, day_length_seconds FROM sunrise_sunset ORDER BY date",
    conn,
)

base_df["year"] = pd.to_datetime(base_df["date"]).dt.year

longest_df = base_df.loc[
    base_df.groupby("year")["day_length_seconds"].idxmax(),
    ["year", "date"],
]
longest_df = longest_df.rename(columns={"date": "longest_day_date"})

shortest_df = base_df.loc[
    base_df.groupby("year")["day_length_seconds"].idxmin(),
    ["year", "date"],
]
shortest_df = shortest_df.rename(columns={"date": "shortest_day_date"})

result_long_short = (
    longest_df.merge(shortest_df, on="year")
    .sort_values("year")
    .reset_index(drop=True)
)

result_long_short

,year,longest_day_date,shortest_day_date
0,2020,2020-06-20,2020-12-21
1,2021,2021-06-20,2021-12-21
2,2022,2022-06-20,2022-12-21
3,2023,2023-06-20,2023-12-22
4,2024,2024-06-20,2024-12-21
5,2025,2025-06-20,2025-12-21
6,2026,2026-03-03,2026-01-01


## 2) Latest Sunrise Time (Per Month)

In [ ]:
query_latest_sunrise = """
SELECT
  strftime('%Y-%m', date) AS year_month,
  MAX(substr(sunrise_ist, 12, 8)) AS latest_sunrise_time_ist
FROM sunrise_sunset
GROUP BY 1
ORDER BY 1;
"""

pd.read_sql_query(query_latest_sunrise, conn)

,year_month,latest_sunrise_time_ist
0,2020-01,01:43:33
1,2020-02,01:41:39
2,2020-03,01:26:18
3,2020-04,01:01:00
4,2020-05,00:39:05
...,...,...
70,2025-11,01:23:42
71,2025-12,01:40:14
72,2026-01,01:43:33
73,2026-02,01:41:29


## 3) Earliest Sunset Time (Per Month)

In [20]:
query_earliest_sunset = """
SELECT
  strftime('%Y-%m', date) AS year_month,
  MIN(substr(sunset_ist, 12, 8)) AS earliest_sunset_time_ist
FROM sunrise_sunset
GROUP BY 1
ORDER BY 1;
"""

pd.read_sql_query(query_earliest_sunset, conn)

,year_month,earliest_sunset_time_ist
0,2020-01,18:13:01
1,2020-02,18:32:15
2,2020-03,18:45:11
3,2020-04,18:53:32
4,2020-05,19:02:01
...,...,...
70,2025-11,18:00:18
71,2025-12,18:00:43
72,2026-01,18:13:20
73,2026-02,18:32:32


In [15]:
conn.close()